# Groupby

## Note préliminaire sur les données

Les données étant volumineuses (1,5 Go une fois chargées dans une feuille de calcul), nous allons définir une ou plusieurs fonctions par exercice pour créer nos diagrammes.

De cette manière, à la sortie de chacune des fonctions, les variables temporaires utilisées seront supprimées et la mémoire pourra être réutilisée.

## Imports

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas
import seaborn as sns

## Chargement des données

Pour ces travaux pratiques, nous allons utiliser des données de transactions immobilières sur la France entière, entre 2014 et 2022.

Ces données proviennent du site d'[open data français](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/). Nous utiliserons là une version retravaillée qui regroupe les transactions, qui provient du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF) sur GitHub.

In [ ]:
!git clone https://github.com/mlambda/dataset-dvf.git

*Vous pouvez procéder au chargement des données avec la fonction [`pandas.read_parquet`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_parquet.html).*

In [ ]:
# Votre code ici

*Une fois les données disponibles dans une feuille de données Pandas, créez une colonne `date` à partir des colonnes `annee_mutation`, `mois_mutation` & `jour_mutation` et de la fonction [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html), puis supprimez ces colonnes pour économiser de la place en mémoire.*

In [ ]:
# Votre code ici

*Enfin, affichez la taille en mémoire ainsi que les types des colonnes de la feuille de données.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
df = pandas.read_parquet("dataset-dvf/dvf-linearized-2014-2022.parquet")

In [ ]:
df["date"] = pandas.to_datetime(dict(year=df.annee_mutation,
                                     month=df.mois_mutation,
                                     day=df.jour_mutation))
df.drop(columns=["annee_mutation", "mois_mutation", "jour_mutation"],
        inplace=True)

Pour une raison de lisibilité par la suite, on met un ```0``` devant les départements à un chiffre

In [ ]:
replacedict = {"code_departement":{ x:"0"+x for x in df.code_departement.unique() if len(x) == 1}}
df.replace(replacedict,inplace=True)


In [ ]:
df.info()

## Nombre de transactions par année

Pour commencer, nous allons créer le décompte par année du nombre de transactions (lignes du jeu de données).

*Utilisez un groupby afin de compter le nombre de transactions par année*

Pour des résultats plus attrayants, utilisez une librairie telle que `seaborn` pour afficher des diagrammes en baton.

In [ ]:
def transactions_by_year() -> None:
  pass  # Votre code ici


transactions_by_year()

### Solution

In [ ]:
def transactions_by_year() -> None:
    data = (df.groupby(df.date.dt.year)
            .valeur_fonciere
            .count())
    fig, ax = plt.subplots(figsize=(10, 10))
    sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
    sns.despine(ax=ax)
    ax.set_title("Décompte des transactions par année")
    ax.set_xlabel("Année")
    ax.set_ylabel("Nombre de transactions")
    ax.get_yaxis().set_major_formatter(matplotlib.ticker.EngFormatter(places=1))
    fig.show()

transactions_by_year()

## Prix moyen par département

Pour commencer, nous allons créer le décompte par année du nombre de transactions (lignes du jeu de données).

*Utilisez un groupby afin de compter le nombre de transactions par année*

Pour des résultats plus attrayants, utilisez une librairie telle que `seaborn` pour afficher des diagrammes en baton.

In [ ]:
def moyenne_par_departement() -> None:
  pass  # Votre code ici


moyenne_par_departement()

### Solution

In [ ]:
def moyenne_par_departement() -> None:
    data = (df.groupby(df.code_departement)
                .valeur_fonciere
                .mean()
                .sort_values(ascending=False))

    fig, ax = plt.subplots(figsize=(30, 10))
    sns.barplot(x=data.index, y=data, order=data.index, ax=ax)
    sns.despine(ax=ax)
    ax.set_title("Montant moyen des transactions par département")
    ax.set_xlabel("Département")
    ax.set_ylabel("Montant moyen des transactions")
    ax.get_yaxis().set_major_formatter(
        matplotlib.ticker.EngFormatter(unit="€", places=1))
    fig.show()

moyenne_par_departement()


In [ ]:
def moyenne_par_departement_std() -> None:
    data = (df.groupby(df.code_departement)
            .valeur_fonciere
            .agg(["min","mean","std","max"])
            .sort_values(by="mean",ascending=False)
            )
    print(data.iloc[:10])
    fig, ax = plt.subplots(figsize=(30, 10))
    sns.barplot(x=data.index, y=data["mean"], order=data.index, ax=ax)
    sns.despine(ax=ax)
    ax.set_title("Montant moyen des transactions par département")
    ax.set_xlabel("Département")
    ax.set_ylabel("Montant moyen des transactions")
    ax.get_yaxis().set_major_formatter(
        matplotlib.ticker.EngFormatter(unit="€", places=1))
    #pour des raisons de lisibilité, on n'affichera l'écart-type divisé par 100
    ax.errorbar(x = data.index, y = data['mean'], yerr=data['std']/100, fmt='none', c= 'black', capsize = 2)
    fig.show()

moyenne_par_departement_std()


## Prix moyen par nature de la mutation

Après avoir regardé les différentes nature de mutation, à l'aide d'un groupby, évaluez les prix moyens pour chacune. N'hésitez pas à afficher d'autres mesures statistiques utiles.

Dans un second temps, regroupez ces descripteurs par départements et affichez les pour les départements ```75``` et ```23```



In [ ]:
# Votre code ici


### Solution

In [ ]:
(df.groupby(["nature_mutation"])
            .valeur_fonciere
            .describe()
            .style.format(precision=0)
)

In [ ]:
data = (df.groupby(["code_departement" , "nature_mutation"])
            .valeur_fonciere
            .agg(["min","mean","std","max"])
            .sort_values(by="mean",ascending=False)
            )
data.loc[["75","23"]].style.format(precision=0)

## Tables pivots

Les tables pivots et les groupby produisent (quasiment) le même résultat si l'on utilise pas l'argument `columns` de pivot_table

In [ ]:
pt = pandas.pivot_table(df, index=["code_departement","nature_mutation"], values="valeur_fonciere", aggfunc=["min","mean","std","max"])
pt.loc[["75","23"],:].style.format(precision=0)

In [ ]:
pt.loc[["75","23"]]['mean']

Observez la différence quand on utilise l'argument `columns` de pivot_table


In [ ]:
pt = pandas.pivot_table(df, index="code_departement", columns="nature_mutation", values="valeur_fonciere", aggfunc=["min","mean","std","max"])
pt.loc[["75","23"],:].style.format(precision=0)

In [ ]:
pt.loc[["75","23"]]['mean']